# How much slower the Python implementation is

This tool removes manual line breaks from Markdown prose. It has two
implementations, one in Python and one in Rust, and for every input the shared
conformance corpus covers they produce identical bytes. The only difference
between them is how long they take to run. This notebook measures that
difference.

There is no single number for it. Three costs make up the total, and they
differ from each other. Starting the program costs a fixed amount, once per
run. Handling one more file adds a small amount for each file. Rewriting the
text inside a file costs more as the file holds more text.

Which of the three accounts for most of the total depends on how the tool is
invoked. For a pre-commit hook on one edited file, starting the program is
almost all of it. For a run over every Markdown file in a repository, the
per-file cost is. For a run over one long document, rewriting the text is.

Every number and every chart below is computed by the code cell above it, so
nothing in this text can contradict the output next to it. A number written
into a sentence would be wrong the next time the notebook runs.

In [1]:
import platform
import re
import subprocess
import sys
from pathlib import Path


def git(*argv: str) -> str:
    """Return the stripped stdout of a git command, or ``''`` if it failed."""
    return subprocess.run(
        ['git', *argv], capture_output=True, text=True, check=False
    ).stdout.strip()


REPO = Path(git('rev-parse', '--show-toplevel'))
PYTHON_CLI = REPO / '.venv/bin/unwrap-markdown-prose-py'
RUST_CLI = REPO / 'target/release/unwrap-markdown-prose-rs'

for path in (PYTHON_CLI, RUST_CLI):
    if not path.exists():
        raise SystemExit(f'missing {path}: run `uv sync` and `cargo build --release`')

# The Python that matters is the one the measured tool runs under, not the one
# executing this notebook, and the two are not always the same: this notebook
# has been executed by an unrelated 3.12 kernel while every timing below still
# came from the interpreter in this repository's virtual environment. Reporting
# the kernel would have credited these timings to an interpreter that never ran
# them. The console script's shebang names the interpreter that actually does.
shebang = PYTHON_CLI.read_text(encoding='utf-8').splitlines()[0]
TOOL_PYTHON = subprocess.run(
    [
        shebang.lstrip('#!').strip(),
        '-c',
        'import platform; print(platform.python_version())',
    ],
    capture_output=True,
    text=True,
    check=False,
).stdout.strip()

# The optimization level is read from Cargo.toml, not copied into this notebook.
# The release profile is set to optimize for binary size, so every Rust timing
# below is the timing of a size-optimized build. Anyone comparing against their
# own build needs to know which setting produced these numbers.
level = re.search(
    r'^\s*opt-level\s*=\s*(\S+)', (REPO / 'Cargo.toml').read_text(), re.MULTILINE
)
OPT_LEVEL = level.group(1).strip('"') if level else 'default'

# Publishing the prebuilt binaries is triggered by a tag. Several rows of the
# final recommendation depend on whether a release exists, so that is read here
# rather than assumed. Local tags are used because this notebook does not make
# network requests, and a tag is what starts the release either way.
TAGS = git('tag', '--list', 'v*').split()

rustc = subprocess.run(
    ['rustc', '--version'], capture_output=True, text=True, check=False
).stdout.strip()

print(f'machine   {platform.machine()}  {platform.system()} {platform.release()}')
print(f'python    {TOOL_PYTHON} runs the measured tool')
if TOOL_PYTHON != platform.python_version():
    print(
        f'          {platform.python_version()} is executing this notebook, '
        'which affects the charts but none of the timings'
    )
print(f'rust      {rustc}')
# `HEAD` is the parent of the commit that carries this output, never the commit
# being measured: the output is part of that commit, so that commit cannot exist
# while the run is happening. The hash is also silent about whether the tree
# matched it, and a re-run is usually prompted by an edit that has not landed
# yet -- which would leave the line naming a tree nobody measured. So the files
# that differ are listed beside it. The charts are left out because this run
# rewrites them itself, which says nothing about what went in.
CHARTS = {
    'docs/benchmarks.svg',
    'docs/benchmarks-bytes.svg',
    'docs/benchmarks-startup.svg',
}
# `diff --name-only` rather than `status --porcelain`: the porcelain format puts
# a two-character status field before each path, and `git()` above strips its
# output, so the first line loses its leading space and any fixed offset then
# eats a character of the first path. This form emits nothing but paths.
DIRTY = sorted(set(git('diff', '--name-only', 'HEAD').splitlines()) - CHARTS)
shown = ', '.join(DIRTY[:4]) + (f' and {len(DIRTY) - 4} more' if len(DIRTY) > 4 else '')
measured = f'{git("rev-parse", "--short", "HEAD")} (the parent of this commit)'
print(f'revision  {measured}')
print(f'          {"plus uncommitted edits to " + shown if DIRTY else "tree clean"}')
BINARY_KB = RUST_CLI.stat().st_size / 1024
tuning = ', optimized for size rather than speed' if OPT_LEVEL in {'z', 's'} else ''
releases = ', '.join(TAGS) if TAGS else 'none yet; see the last cell'

print(f'binary    {BINARY_KB:.0f} KB at opt-level {OPT_LEVEL!r}{tuning}')
print(f'releases  {releases}')

machine   arm64  Darwin 23.5.0
python    3.10.18 runs the measured tool
rust      rustc 1.86.0 (05f9846f8 2025-03-31)
revision  6ef50b2 (the parent of this commit)
          plus uncommitted edits to docs/benchmarks.ipynb
binary    344 KB at opt-level 'z', optimized for size rather than speed
releases  v0.0.1


## Method

Each measurement times one complete run of the program, because that is what a
pre-commit hook or a CI step actually costs. Three warm-up runs are discarded,
and the runs after them are timed.

**Both the minimum and the median are reported.** The minimum is the better
estimate of what the program costs, because it is the run least affected by
other work happening on the machine at the same time. The median is printed
next to it so that the difference between the two is visible. A large
difference means the machine was busy, and a cell below checks for this and
reports it.

**The cost of starting a process is measured, not subtracted.** Every number
here includes the time to start a process: the operating system call that
creates it, the one that loads the program, and the pipe setup that this
notebook's own timing loop adds. That time is very small compared to the Python
total, and large compared to the Rust total. Dividing one by the other
therefore understates how much faster Rust is.

Subtracting that time would be less accurate than reporting it. The next cell
times three programs that do almost nothing, and they differ from each other by
more than the entire Rust runtime, so there is no single correct value to
subtract. Instead the cheapest of them is reported as a floor, drawn on the
charts, and used to mark any ratio whose smaller number is close to that floor.
Those ratios are labeled as lower bounds: the real difference is larger than
the number shown.

**The two implementations are compared inside the benchmark.** For every run,
the timing loop records a hash of the output and the exit code, then compares
both between Python and Rust. Without this check, a program that rejected its
arguments and exited immediately would record a very fast time and look like an
improvement. Recording a hash for every run, rather than keeping only the last
one, also detects a file being modified while the benchmark is reading it.

In [2]:
import hashlib
import statistics
import time
from dataclasses import dataclass

WARMUP, REPS = 3, 30

# How far above the minimum the median may be, as a percentage, before a
# measurement is reported as unreliable. Taken from the startup rows, which are
# the most repeatable measurements here and are a few percent apart on an
# otherwise idle machine.
SPREAD_LIMIT = 15.0

# How close to the floor the smaller of two times may be before their ratio is
# reported as a lower bound instead of a value.
FLOOR_FACTOR = 2.0

# Marks a ratio as a lower bound. Defined here as a name because an escape
# sequence cannot appear inside an f-string expression before Python 3.12.
BOUND = '\u2265'


@dataclass(frozen=True, slots=True)
class Timing:
    """Every sample from one measured command, and what those runs returned."""

    samples: tuple[float, ...]
    codes: frozenset[int]
    # One hash per repetition, rather than one saved copy of the output. If two
    # runs of the same command return different bytes, something changed the
    # files while the benchmark was reading them. Saving only one copy of the
    # output cannot show that, because the copy saved is the last run.
    digests: frozenset[str]

    @property
    def min(self) -> float:
        """Return the run least affected by other work on the machine."""
        return min(self.samples)

    @property
    def median(self) -> float:
        """Return the middle sample, which is raised by a busy machine."""
        return statistics.median(self.samples)

    @property
    def spread(self) -> float:
        """Return how far the median is above the minimum, as a percentage."""
        return (self.median - self.min) / self.min * 100

    @property
    def spawn_bound(self) -> bool:
        """Return whether this time is mostly the cost of starting a process."""
        return self.min < SPAWN_FLOOR * FLOOR_FACTOR


subprocess_run = subprocess.run
perf_counter = time.perf_counter
sha256 = hashlib.sha256


def measure(argv: list[str], reps: int = REPS) -> Timing:
    """Time ``reps`` complete runs of ``argv``, after ``WARMUP`` discarded runs."""
    for _ in range(WARMUP):
        subprocess_run(argv, capture_output=True, check=False)
    samples, codes, digests = [], set(), set()
    append_to_samples = samples.append
    add_to_codes = codes.add
    add_to_digests = digests.add
    for _ in range(reps):
        started = perf_counter()
        done = subprocess_run(argv, capture_output=True, check=False)
        append_to_samples((perf_counter() - started) * 1000)
        add_to_codes(done.returncode)
        add_to_digests(sha256(done.stdout).hexdigest())
    return Timing(tuple(samples), frozenset(codes), frozenset(digests))


# Three programs that do almost nothing, timed through the same loop as
# everything else. The cheapest is used as the floor. All three are printed
# because the differences between them are the reason none of them is
# subtracted from the measurements.
REFERENCES = ['/bin/echo', 'x'], ['/usr/bin/true'], ['/usr/bin/printf', '']

reference = {__: measure(argv) for argv in REFERENCES if Path(__ := argv[0]).exists()}
FLOOR = min(reference.values(), key=lambda timing: timing.min)
SPAWN_FLOOR = FLOOR.min

for name, timing in sorted(reference.items(), key=lambda item: item[1].min):
    print(f'{name:<18} {timing.min:>6.2f} ms   (median {timing.median:>5.2f} ms)')
spread = max(t.min for t in reference.values()) - SPAWN_FLOOR
print(f'\nfloor              {SPAWN_FLOOR:>6.2f} ms, the cheapest of the three')
print(f'they differ by     {spread:>6.2f} ms, which is why none of them is subtracted')

/usr/bin/true        1.11 ms   (median  1.28 ms)
/bin/echo            1.16 ms   (median  1.60 ms)
/usr/bin/printf      1.19 ms   (median  1.35 ms)

floor                1.11 ms, the cheapest of the three
they differ by       0.08 ms, which is why none of them is subtracted


In [3]:
import json

scratch = Path('/tmp/markdown-prose-bench')
scratch.mkdir(exist_ok=True)

# One paragraph broken across several lines is the unit of work this tool
# exists to undo.
PARAGRAPH = 'A paragraph that has been hard\nwrapped across three\nseparate lines.\n\n'
(scratch / 'tiny.md').write_text(PARAGRAPH)

tracked = subprocess.run(
    ['git', 'ls-files', '*.md'], cwd=REPO, capture_output=True, text=True, check=True
).stdout.split()
(scratch / 'tracked.txt').write_text(
    '\n'.join(str(REPO / name) for name in tracked) + '\n'
)

# A generated set of files, so that the number of files can be varied
# independently of this repository.
bulk = scratch / 'bulk'
bulk.mkdir(exist_ok=True)
for index in range(2000):
    (bulk / f'{index}.md').write_text(PARAGRAPH * 3)

# One large document. Startup is a negligible part of the time taken to process
# it, so what remains is the speed of the transform itself.
large = scratch / 'large.md'
large.write_text(PARAGRAPH * 60000)

# Passed explicitly because this notebook runs from the docs directory, and the
# tool looks for .unwrapignore in the directory it is run from. Without it the
# benchmark would process the corpus directory, which holds test fixtures
# rather than prose: around 200 expected-output files of a few dozen bytes
# each, plus one file that is deliberately not valid UTF-8 and that the tool
# correctly refuses to read. Those files are not this repository's Markdown,
# and an average taken over them is not this repository's file size.
IGNORE = '--ignore-file', str(REPO / '.unwrapignore')

# Which files count as in scope is determined by running the tool rather than
# reimplemented here, because the tool defines the ignore rules.
scope = json.loads(
    subprocess.run(
        [
            str(PYTHON_CLI),
            '--files-from',
            str(scratch / 'tracked.txt'),
            *IGNORE,
            '--json',
        ],
        capture_output=True,
        text=True,
        check=True,
    ).stdout
)
in_scope = [Path(entry['path']) for entry in scope['files']]
sizes = sorted(path.stat().st_size for path in in_scope)
REPO_MEAN_BYTES = sum(sizes) / len(sizes)
len_in_scope, len_tracked = len(in_scope), len(tracked)

print(
    f'{len_tracked} tracked Markdown files, {len_in_scope} in scope and '
    f'{len_tracked - len_in_scope} excluded as test fixtures'
)
average = f'mean {REPO_MEAN_BYTES:,.0f} bytes per file'
print(f'  in scope: {sum(sizes) / 1024:.0f} KB, {average}')
print(f'generated file: {len(PARAGRAPH) * 3} bytes')
print(f'large document: {large.stat().st_size / 1e6:.1f} MB')

232 tracked Markdown files, 7 in scope and 225 excluded as test fixtures
  in scope: 203 KB, mean 29,717 bytes per file
generated file: 207 bytes
large document: 4.1 MB


In [4]:
from typing import Literal

from IPython.display import Markdown, display


@dataclass(frozen=True, slots=True)
class Scenario:
    """One row of the table below: what was processed, and how long each took."""

    label: str
    python: Timing
    rust: Timing

    @property
    def timings(self) -> tuple[tuple[str, Timing], tuple[str, Timing]]:
        """Return each timing with the name of the program that produced it."""
        return ('python', self.python), ('rust', self.rust)


def file_list(count: int) -> tuple[Literal['--files-from'], str]:
    """Return arguments naming the first ``count`` of the generated files."""
    path = scratch / f'bulk-{count}.txt'
    path.write_text(
        '\n'.join(str(bulk / f'{index}.md') for index in range(count)) + '\n'
    )
    return '--files-from', str(path)


# One run over the 4 MB document takes about a second in Python, so that row
# uses fewer repetitions than the others. Stated here rather than left for a
# reader to work out from how long the cell takes.
LARGE_REPS = 10

scenarios = (
    ('one file', [str(scratch / 'tiny.md')], REPS),
    ('a typical commit: 5 files', file_list(5), REPS),
    (
        f'this repository: {len(in_scope)} prose files at '
        f'{REPO_MEAN_BYTES / 1024:.0f} KB each',
        ['--files-from', str(scratch / 'tracked.txt'), *IGNORE],
        REPS,
    ),
    ('a large repository: 2000 files', file_list(2000), REPS),
    (f'one {large.stat().st_size / 1e6:.0f} MB document', [str(large)], LARGE_REPS),
)

results: list[Scenario] = []
append_to_results = results.append
for label, args, reps in scenarios:
    append_to_results(
        Scenario(
            label=label,
            python=measure([str(PYTHON_CLI), *args, '--json'], reps),
            rust=measure([str(RUST_CLI), *args, '--json'], reps),
        )
    )


def ratio_cell(row: Scenario) -> str:
    """Return Python divided by Rust, marked as a bound when Rust is near the floor."""
    value = row.python.min / row.rust.min
    mark = BOUND if row.rust.spawn_bound else ''
    return f'{mark}{value:.1f}x'


table = [
    '| what is being processed | Python min / median | Rust min / median | Python is |',
    '| -- | --: | --: | --: |',
]
append_to_table = table.append
for row in results:
    append_to_table(
        f'| {row.label} '
        f'| {row.python.min:.1f} / {row.python.median:.1f} ms '
        f'| {row.rust.min:.1f} / {row.rust.median:.1f} ms '
        f'| {ratio_cell(row)} slower |'
    )
append_to_table('')
append_to_table(
    f'{BOUND} means the Rust time is within {FLOOR_FACTOR:.0f} times the '
    f'{SPAWN_FLOOR:.2f} ms floor, so most of what it measures is the cost of starting '
    'any process at all. Those ratios are lower bounds, and the real difference is '
    'larger.'
)
display(Markdown(chr(10).join(table)))

# Three separate questions, because they have three separate answers, and only
# the first of them can invalidate a recommendation.
#
#   AGREE    did the two implementations return the same bytes and the same
#            exit code? This is the assumption the whole document depends on.
#   STEADY   did each command return the same result on every repetition? It
#            may not, because the file list comes from the working directory,
#            and a file edited while the benchmark runs appears here as a read
#            error in one repetition out of thirty.
#   CLEAN    did every run exit with status 0? A consistent non-zero status
#            means the tool reported a file it could not read, which is a fact
#            about the files rather than about either implementation.
AGREE = all(
    row.python.digests == row.rust.digests and row.python.codes == row.rust.codes
    for row in results
)
unsteady = [
    (row.label, name)
    for row in results
    for name, timing in row.timings
    if len(timing.digests) > 1 or len(timing.codes) > 1
]
CLEAN = all(
    timing.codes == frozenset({0}) for row in results for _, timing in row.timings
)
# A time that measures more than process startup is held to SPREAD_LIMIT. A time
# that is mostly startup is reported separately and held to nothing: the table
# already marks its ratio as a lower bound, and asking a reader to measure it
# again reproduces the same width, which was checked across three runs.
#
# The reference programs' own spread is reported as the evidence for that rather
# than used as the tolerance. It is three sub-millisecond measurements, and the
# widest of them has ranged from single digits to over 40% between runs of this
# notebook, so a tolerance drawn from it would loosen every row on the run where
# one reference program happened to stutter.
FLOOR_SPREAD = max(timing.spread for timing in reference.values())

over = [
    (row.label, name, timing)
    for row in results
    for name, timing in row.timings
    if timing.spread > SPREAD_LIMIT
]
noisy = [entry for entry in over if not entry[2].spawn_bound]
startup_bound = [entry for entry in over if entry[2].spawn_bound]

if AGREE:
    print('Both implementations returned identical bytes and identical exit codes for')
    print('every row above. The only difference between them here is speed.')
else:
    print('The implementations returned different results. Ignore every')
    print('recommendation below.')
    for row in results:
        if row.python.digests != row.rust.digests:
            print(f'     different output: {row.label}')
        if row.python.codes != row.rust.codes:
            print(f'     different exit status: {row.label}')

if unsteady:
    print('\nA command did not return the same result on every repetition, so a file')
    print('changed while the benchmark was running. The timings are still usable, but')
    print('these rows are worth measuring again:')
    for label, impl in unsteady:
        print(f'     {impl:<7} {label}')
elif not CLEAN:
    print('\nEvery repetition of one row exited with a non-zero status, identically in')
    print('both implementations. The tool reported a file it could not read. That is a')
    print('fact about the files, not a difference between the implementations.')
    for row in results:
        codes = row.python.codes | row.rust.codes
        if codes != frozenset({0}):
            print(f'     {row.label}: exit {sorted(codes)}')

if noisy:
    print(f'\nIn these rows the median is more than {SPREAD_LIMIT:.0f}% above the')
    print('minimum, and the row measures more than process startup, so the')
    print('machine was busy. Measure them again:')
    for label, impl, timing in noisy:
        print(f'     {impl:<7} {label:<44} +{timing.spread:.0f}%')
else:
    print('\nEvery row that measures more than starting a process is within')
    print(f'{SPREAD_LIMIT:.0f}% of its own minimum.')

if startup_bound:
    print(f'\nThese are wider than {SPREAD_LIMIT:.0f}%, and are expected to be.')
    print(f'Their time is mostly the {SPAWN_FLOOR:.2f} ms of starting a process,')
    print(f'which varied by {FLOOR_SPREAD:.0f}% this run while doing nothing. The')
    print(f'table marks their ratios {BOUND} for the same reason:')
    for label, impl, timing in startup_bound:
        print(f'     {impl:<7} {label:<44} +{timing.spread:.0f}%')

| what is being processed | Python min / median | Rust min / median | Python is |
| -- | --: | --: | --: |
| one file | 25.0 / 25.7 ms | 1.7 / 1.9 ms | ≥14.9x slower |
| a typical commit: 5 files | 25.2 / 26.8 ms | 1.4 / 1.5 ms | ≥17.6x slower |
| this repository: 7 prose files at 29 KB each | 65.0 / 66.4 ms | 3.6 / 4.3 ms | 18.2x slower |
| a large repository: 2000 files | 202.0 / 206.3 ms | 33.5 / 35.6 ms | 6.0x slower |
| one 4 MB document | 1047.9 / 1058.0 ms | 68.6 / 70.2 ms | 15.3x slower |

≥ means the Rust time is within 2 times the 1.11 ms floor, so most of what it measures is the cost of starting any process at all. Those ratios are lower bounds, and the real difference is larger.

Both implementations returned identical bytes and identical exit codes for
every row above. The only difference between them here is speed.

In these rows the median is more than 15% above the
minimum, and the row measures more than process startup, so the
machine was busy. Measure them again:
     rust    this repository: 7 prose files at 29 KB each +21%


## What the time is spent on

The table above combines two separate costs: a fixed cost paid once per run,
and a cost paid for each file. Telling them apart is what makes it possible to
recommend one implementation per way of using the tool.

The next cell measures the fixed cost directly. It times the cheapest process
this machine can start, a Python interpreter that does nothing, the same
interpreter after importing this tool, and then each of the two programs on a
single file.

The difference between the two interpreter rows is the cost of importing the
tool. The difference between the last two rows is what one run of the Rust
program saves. The first row is the cost that neither implementation can go
below, because it is what starting any process costs.

In [5]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.axes import Axes
from matplotlib.figure import Figure

# Element ids are derived from this value rather than from memory addresses, so
# two runs over the same numbers produce the same file.
mpl.rcParams['svg.hashsalt'] = 'markdown-prose-hooks'


def hide_spines(*all_axes: Axes) -> None:
    """Remove the top and right borders from each chart."""
    for axes in all_axes:
        axes.spines['top'].set_visible(False)
        axes.spines['right'].set_visible(False)


def save_chart(figure: Figure, name: str) -> None:
    """Write ``figure`` to the docs directory as the bytes that get committed.

    Two things would otherwise change the file on every run, whatever was
    measured. matplotlib writes the current date into the SVG metadata, and it
    leaves trailing spaces on a few hundred lines, which this repository
    removes with its own hook when the file is committed. Left as they are, the
    committed chart is not the file this notebook wrote, running the notebook
    always leaves uncommitted changes, and a real change to a measurement is
    hard to see among the changes that mean nothing.
    """
    path = REPO / 'docs' / name
    figure.savefig(path, metadata={'Date': None})
    plt.close(figure)
    text = path.read_text(encoding='utf-8')
    path.write_text(
        chr(10).join(line.rstrip() for line in text.split(chr(10))),
        encoding='utf-8',
    )
    print(f'chart written to docs/{name}')


floor_rows = {
    'cheapest process': FLOOR,
    'Python interpreter, doing nothing': measure([sys.executable, '-c', 'pass']),
    'Python interpreter, after import': measure(
        [sys.executable, '-c', 'import markdown_prose_hooks.unwrap']
    ),
    'Python program, one file': measure(
        [str(PYTHON_CLI), str(scratch / 'tiny.md'), '--json']
    ),
    'Rust program, one file': measure(
        [str(RUST_CLI), str(scratch / 'tiny.md'), '--json']
    ),
}
for label, timing in floor_rows.items():
    print(f'{label:<36} {timing.min:>6.2f} ms   (median {timing.median:>6.2f} ms)')

python_startup = floor_rows['Python program, one file'].min
rust_startup = floor_rows['Rust program, one file'].min
saving = python_startup - rust_startup

print(f'\nPython takes {python_startup / rust_startup:.1f} times as long to start')
if rust_startup > SPAWN_FLOOR:
    corrected = (python_startup - SPAWN_FLOOR) / (rust_startup - SPAWN_FLOOR)
    print(f'With the floor removed from both, it would be {corrected:.1f} times.')
    print(
        f'   That figure is shown but not used. The Rust time is only '
        f'{rust_startup / SPAWN_FLOOR:.1f} times'
    )
    print('   the floor, so removing the floor divides by the small difference between')
    print('   two similar numbers, and the result changes greatly when the floor')
    print('   changes slightly. Treat the figure above as a minimum instead.')
else:
    print('   The Rust program was as fast as the cheapest process this machine can')
    print('   start, so its own startup cost is below what this method can measure.')
    print('   The figure above is a minimum.')

print(f'\nEach run of the Rust program saves {saving:.1f} ms.')
print(f'Over a hundred runs a day that is {saving * 100 / 1000:.1f} seconds.')
print('   Subtracting is reliable here even though dividing is not. The cost of')
print('   starting a process is the same in both numbers, so it cancels out when')
print('   they are subtracted.')

figure, axes = plt.subplots(figsize=(8, 3.2))
labels = list(floor_rows)
axes.barh(
    labels,
    [floor_rows[label].min for label in labels],
    color=['#999999', '#bbbbbb', '#bbbbbb', '#1f77b4', '#ff7f0e'],
)
axes.axvline(SPAWN_FLOOR, color='#444444', linestyle='--', linewidth=1)
axes.set_xscale('log')
axes.set_xlabel('time (milliseconds, logarithmic scale)')
axes.set_title('The dashed line is what starting any process costs')
axes.invert_yaxis()
for index, label in enumerate(labels):
    axes.text(
        floor_rows[label].min * 1.15,
        index,
        f'{floor_rows[label].min:.2f} ms',
        va='center',
        fontsize=9,
    )
axes.set_xlim(right=max(t.min for t in floor_rows.values()) * 3)
axes.grid(alpha=0.3, axis='x')
hide_spines(axes)
figure.tight_layout()
print()
save_chart(figure, 'benchmarks-startup.svg')

cheapest process                       1.11 ms   (median   1.28 ms)
Python interpreter, doing nothing     12.93 ms   (median  14.69 ms)
Python interpreter, after import      24.49 ms   (median  25.02 ms)
Python program, one file              25.93 ms   (median  26.91 ms)
Rust program, one file                 1.72 ms   (median   1.92 ms)

Python takes 15.1 times as long to start
With the floor removed from both, it would be 40.9 times.
   That figure is shown but not used. The Rust time is only 1.5 times
   the floor, so removing the floor divides by the small difference between
   two similar numbers, and the result changes greatly when the floor
   changes slightly. Treat the figure above as a minimum instead.

Each run of the Rust program saves 24.2 ms.
Over a hundred runs a day that is 2.4 seconds.
   Subtracting is reliable here even though dividing is not. The cost of
   starting a process is the same in both numbers, so it cancels out when
   they are subtracted.

chart written 

![The dashed line is what starting any process costs](benchmarks-startup.svg)

## Separating the fixed cost from the cost per file

Running the tool once over an increasing number of files separates the two
costs. In the left chart, where a line begins is the fixed cost, and how
steeply it rises is the cost per file. The right chart shows how many times
slower Python is at each number of files.

Both axes on the left are logarithmic because the number of files spans three
orders of magnitude. On ordinary axes, six of the eight measurements would fall
within the first quarter of the chart, where the fixed cost cannot be read.

In [6]:
# Fewer repetitions than the table above. This cell and the next one exist to
# show the shape of a curve rather than to produce a figure quoted elsewhere,
# and the shape is already stable at this number of repetitions.
SWEEP_REPS = 15

counts = 1, 10, 50, 100, 250, 500, 1000, 2000
curve = {'Python': [], 'Rust': []}
python_curve, rust_curve = curve['Python'], curve['Rust']
append_to_python_curve = python_curve.append
append_to_rust_curve = rust_curve.append
for count in counts:
    args = file_list(count)
    append_to_python_curve(measure([str(PYTHON_CLI), *args, '--json'], SWEEP_REPS).min)
    append_to_rust_curve(measure([str(RUST_CLI), *args, '--json'], SWEEP_REPS).min)

figure, (left, right) = plt.subplots(1, 2, figsize=(11, 4.2))
plot_left = left.plot
for name, series in curve.items():
    plot_left(counts, series, marker='o', label=name)
left.axhline(SPAWN_FLOOR, color='#444444', linestyle='--', linewidth=1)
left.text(counts[0], SPAWN_FLOOR * 1.15, 'floor', fontsize=8, color='#444444')
left.set_xscale('log')
left.set_yscale('log')
left.set_xlabel('files processed in one run')
left.set_ylabel('time (milliseconds)')
left.set_title('Where a line starts is the fixed cost')
left.legend()
left.grid(alpha=0.3, which='both')

ratios = [p / r for p, r in zip(python_curve, rust_curve, strict=True)]
right.plot(counts, ratios, marker='o', color='#2ca02c')
right.set_xscale('log')
right.set_ylim(bottom=0)
right.set_xlabel('files processed in one run')
right.set_ylabel('times slower')
right.set_title('Python, relative to Rust')
right.grid(alpha=0.3, which='both')
hide_spines(left, right)

# The chart is written next to the notebook rather than embedded in it, and as
# SVG rather than PNG. An embedded PNG is stored as base64 text, which the two
# spell-checking hooks read as prose: measured on thirty different versions of
# this chart, codespell reported a misspelling in eighteen of them, because
# base64 splits into thousands of short letter sequences and some of them match
# dictionary entries. The same thirty charts written as SVG produced no
# findings from either hook. A PNG file would be worse still, because this
# repository routes `*.png` through Git LFS, so the chart would be committed as
# a pointer and appear broken to anyone cloning without git-lfs installed.
figure.tight_layout()
save_chart(figure, 'benchmarks.svg')

# A straight line between the first and last points is only a fair summary if
# the measurements between them are also close to that line. Every consecutive
# pair is printed so that a reader can see whether they are, rather than being
# asked to assume it.
count_marginal, count_intercept = {}, {}
print(f'{"":<8} {"fixed cost":>11} {"per file":>12}   cost per file between each pair')
for name, series in curve.items():
    pairs = [
        (series[i + 1] - series[i]) / (counts[i + 1] - counts[i]) * 1000
        for i in range(len(counts) - 1)
    ]
    count_intercept[name] = series[0]
    count_marginal[name] = (series[-1] - series[0]) / (counts[-1] - counts[0]) * 1000
    print(
        f'{name:<8} {series[0]:>8.1f} ms {count_marginal[name]:>8.1f} us   '
        + ' '.join(f'{value:.0f}' for value in pairs)
    )
print('\nThe leftmost pairs are the least reliable, because they are differences')
print('between measurements only a few milliseconds apart. They stabilize once the')
print('per-file work is large enough to measure, and that is what makes the single')
print('figure in the third column a fair summary rather than a straight line drawn')
print('through a curve.')
print(
    f'\nPython is {ratios[0]:.1f} times slower at {counts[0]} file and '
    f'{ratios[-1]:.1f} times slower at {counts[-1]} files.'
)

chart written to docs/benchmarks.svg
          fixed cost     per file   cost per file between each pair
Python       26.8 ms     87.8 us   91 85 92 89 87 87 88
Rust          1.7 ms     16.1 us   26 13 11 15 16 16 17

The leftmost pairs are the least reliable, because they are differences
between measurements only a few milliseconds apart. They stabilize once the
per-file work is large enough to measure, and that is what makes the single
figure in the third column a fair summary rather than a straight line drawn
through a curve.

Python is 15.3 times slower at 1 file and 6.0 times slower at 2000 files.


![Where a line starts is the fixed cost](benchmarks.svg)

## What the cost per file depends on

The cost of one more file is not a fixed number, and it is not a property of
the tool. Opening a file is a request to the operating system, and costs about
the same in both languages. Reading its contents and rewriting the paragraphs
is the work the two implementations do differently. So how much slower Python
is per file depends on how much text an average file contains.

The next cell measures that. For each file size it times two runs, one over a
small number of files and one over a large number, and subtracts the first from
the second. Subtracting removes the fixed startup cost exactly, without having
to estimate it. That matters most for small files, where startup would
otherwise be most of what was measured.

The result is a single curve. At one end, processing a file costs little more
than opening it. At the other end, the cost is almost entirely the work of
rewriting the text. Any figure quoted for the difference per file is one point
on this curve, and which point applies to a repository depends on the
Markdown files in it. The marked point is the value for this repository.

In [7]:
import numpy as np

# Two numbers of files for each file size, and the difference between them. The
# fixed startup cost is identical in both, so subtracting removes it exactly
# and nothing here needs an estimate of it.
SIZE_REPS = 8
LOW, HIGH = 100, 500
paragraph_counts = 1, 3, 10, 30, 100


def listing(
    folder: Path, count: int
) -> tuple[Literal['--files-from'], str, Literal['--json']]:
    """Return arguments naming ``count`` files from ``folder``, asking for JSON."""
    path = scratch / f'{folder.name}-{count}.txt'
    path.write_text(
        '\n'.join(str(folder / f'{index}.md') for index in range(count)) + '\n'
    )
    return '--files-from', str(path), '--json'


def marginal_us(cli: Path, folder: Path) -> float:
    """Return the microseconds one more file from ``folder`` costs ``cli``.

    The fixed startup cost is identical in both measurements, so subtracting
    one from the other removes it exactly rather than by estimate.
    """
    low = measure([str(cli), *listing(folder, LOW)], SIZE_REPS).min
    high = measure([str(cli), *listing(folder, HIGH)], SIZE_REPS).min
    return (high - low) / (HIGH - LOW) * 1000


sweep = {'bytes': [], 'Python': [], 'Rust': []}
sweep_bytes = sweep['bytes']
sweep_python = sweep['Python']
sweep_rust = sweep['Rust']
append_to_sweep_bytes = sweep_bytes.append
append_to_sweep_python = sweep_python.append
append_to_sweep_rust = sweep_rust.append
for paragraphs in paragraph_counts:
    folder = scratch / f'size-{paragraphs}'
    folder.mkdir(exist_ok=True)
    for index in range(HIGH):
        (folder / f'{index}.md').write_text(PARAGRAPH * paragraphs)
    append_to_sweep_bytes(len(PARAGRAPH) * paragraphs)
    append_to_sweep_python(marginal_us(PYTHON_CLI, folder))
    append_to_sweep_rust(marginal_us(RUST_CLI, folder))

size_ratios = [p / r for p, r in zip(sweep_python, sweep_rust, strict=True)]

# Interpolated on a logarithmic scale, because the sizes measured are spaced
# logarithmically.
REPO_RATIO = float(
    np.interp(np.log10(REPO_MEAN_BYTES), np.log10(sweep['bytes']), size_ratios)
)
CLAMPED = not sweep_bytes[0] <= REPO_MEAN_BYTES <= sweep_bytes[-1]
RATIO_TEXT = f'{REPO_RATIO:.1f} times or more' if CLAMPED else f'{REPO_RATIO:.1f} times'

# Kept within the range measured, so that a mean larger than the largest size
# measured is marked at the end of the curve rather than stretching the chart
# into empty space.
marker_x = min(max(REPO_MEAN_BYTES, sweep_bytes[0]), sweep_bytes[-1])

figure, axes = plt.subplots(figsize=(8, 4.5))
axes.plot(sweep_bytes, size_ratios, marker='o', color='#2ca02c')
axes.axvline(marker_x, color='#d62728', linestyle='--', linewidth=1)
axes.annotate(
    f'this repository\n{REPO_MEAN_BYTES:,.0f} bytes per file, {RATIO_TEXT}',
    xy=(marker_x, REPO_RATIO),
    xytext=(-16, -78) if CLAMPED else (16, -34),
    textcoords='offset points',
    horizontalalignment='right' if CLAMPED else 'left',
    fontsize=9,
    color='#d62728',
    arrowprops={'arrowstyle': '->', 'color': '#d62728'},
)
axes.set_xscale('log')
axes.set_ylim(bottom=0)
axes.set_xlabel(
    f'bytes per file (logarithmic scale; each point subtracts {LOW} files from {HIGH})'
)
axes.set_ylabel('times slower, per file')
axes.set_title('How much slower Python is per file depends on the text in it')
axes.grid(alpha=0.3, which='both')
hide_spines(axes)
figure.tight_layout()
save_chart(figure, 'benchmarks-bytes.svg')

table = [
    '| bytes per file | Python | Rust | Python is |',
    '| --: | --: | --: | --: |',
]
append_to_table = table.append
for size, python_us, rust_us, value in zip(
    sweep_bytes, sweep_python, sweep_rust, size_ratios, strict=True
):
    append_to_table(
        f'| {size} | {python_us:.1f} us per file | {rust_us:.1f} us per file '
        f'| {value:.1f}x slower |'
    )
append_to_table('')
append_to_table(
    f'Across the {sweep["bytes"][-1] // sweep["bytes"][0]}-fold range of file sizes '
    f'measured, Python is between {min(size_ratios):.1f} and {max(size_ratios):.1f} '
    f'times slower per file. Files in this repository average '
    f'{REPO_MEAN_BYTES:,.0f} bytes, where the figure is **{RATIO_TEXT}**'
    + (
        '. That is beyond the largest size measured, but the curve has already '
        'leveled off by then, so the true figure is close to the value at the end '
        'of the curve rather than far above it.'
        if CLAMPED
        else '.'
    )
)
display(Markdown(chr(10).join(table)))

# The previous cell measured this same quantity at one file size, because the
# files it used are 207 bytes each. Two independent measurements of one
# quantity are worth comparing rather than assuming they agree.
common = sweep['bytes'].index(len(PARAGRAPH) * 3)
print(
    f'Checked against the previous cell at {sweep["bytes"][common]} bytes per file, '
    'where both measured the same thing:'
)
for name in ('Python', 'Rust'):
    here, there = sweep[name][common], count_marginal[name]
    print(
        f'  {name:<7} {here:>6.1f} us here, {there:>6.1f} us there, a difference of '
        f'{abs(here - there) / there * 100:.0f}%'
    )

chart written to docs/benchmarks-bytes.svg


| bytes per file | Python | Rust | Python is |
| --: | --: | --: | --: |
| 69 | 62.8 us per file | 12.8 us per file | 4.9x slower |
| 207 | 90.7 us per file | 17.1 us per file | 5.3x slower |
| 690 | 210.3 us per file | 23.5 us per file | 9.0x slower |
| 2070 | 565.7 us per file | 49.6 us per file | 11.4x slower |
| 6900 | 1754.4 us per file | 127.0 us per file | 13.8x slower |

Across the 100-fold range of file sizes measured, Python is between 4.9 and 13.8 times slower per file. Files in this repository average 29,717 bytes, where the figure is **13.8 times or more**. That is beyond the largest size measured, but the curve has already leveled off by then, so the true figure is close to the value at the end of the curve rather than far above it.

Checked against the previous cell at 207 bytes per file, where both measured the same thing:
  Python    90.7 us here,   87.8 us there, a difference of 3%
  Rust      17.1 us here,   16.1 us there, a difference of 6%


![How much slower Python is per file depends on the text in it](benchmarks-bytes.svg)

## What each implementation costs to install

Everything above measures how long the tool takes to run. Choosing between the
two also depends on what each costs to set up, because `pre-commit` builds an
environment for a hook the first time it runs it, and again whenever the pinned
revision changes.

The next cell measures both. For Python it creates a virtual environment and
installs the package into it, which is what `pre-commit` does for a
`language: python` hook. For Rust it compiles and installs the program, which
is what it does for a `language: rust` hook. Each is timed into an empty
directory, so nothing is reused between one run and the next.

One case is deliberately not measured here: a machine with no Rust toolchain.
There `pre-commit` downloads and installs a toolchain first, which costs far
more than anything on this page, and it is why the rule in the README turns on
whether cargo is already installed.

In [8]:
import os
import shutil
from collections.abc import Callable

# Three cold runs each. Fewer than the timings above because each run creates a
# whole environment, and because the numbers are seconds apart rather than
# milliseconds apart, so a small disturbance changes them proportionally less.
INSTALL_REPS = 3
installs = scratch / 'installs'


@dataclass(frozen=True, slots=True)
class InstallCost:
    """Seconds taken to create one environment, over several cold runs."""

    samples: tuple[float, ...]

    @property
    def min(self) -> float:
        """Return the fastest of the cold runs."""
        return min(self.samples)

    @property
    def median(self) -> float:
        """Return the middle cold run."""
        return statistics.median(self.samples)


subprocess_run = subprocess.run


def python_environment(target: Path) -> None:
    """Create a virtual environment in ``target`` and install this package."""
    subprocess_run(
        [sys.executable, '-m', 'venv', str(target)], check=True, capture_output=True
    )
    subprocess_run(
        [
            str(target / 'bin/pip'),
            'install',
            '--quiet',
            '--disable-pip-version-check',
            str(REPO),
        ],
        check=True,
        capture_output=True,
    )


def rust_environment(target: Path) -> None:
    """Compile the Rust program and install it into ``target``."""
    subprocess_run(
        [
            'cargo',
            'install',
            '--quiet',
            '--locked',
            '--root',
            str(target),
            '--path',
            str(REPO),
        ],
        check=True,
        capture_output=True,
        env={**os.environ, 'CARGO_TARGET_DIR': str(target / 'build')},
    )


rmtree = shutil.rmtree
perf_counter = time.perf_counter


def time_environment(setup: Callable[[Path], None]) -> InstallCost:
    """Time ``setup`` into an empty directory, ``INSTALL_REPS`` times."""
    samples = []
    append_to_samples = samples.append
    for index in range(INSTALL_REPS):
        target = installs / f'{setup.__name__}-{index}'
        target.parent.mkdir(parents=True, exist_ok=True)
        rmtree(target, ignore_errors=True)
        started = perf_counter()
        setup(target)
        append_to_samples(perf_counter() - started)
        rmtree(target, ignore_errors=True)
    return InstallCost(tuple(samples))


python_install = time_environment(python_environment)
rust_environment_cost = time_environment(rust_environment)

print(
    f'Python environment, venv plus pip install   {python_install.min:.2f} s   '
    f'(median {python_install.median:.2f} s)'
)
print(
    f'Rust environment, cargo install            '
    f'{rust_environment_cost.min:.2f} s   '
    f'(median {rust_environment_cost.median:.2f} s)'
)

# Below this fraction the two are treated as equal. Both figures are seconds
# spent creating files and compiling, which varies with what else the machine is
# doing, and repeated runs of this cell have put each one ahead of the other.
# Reporting a winner from a difference this small would claim a precision the
# measurement does not have.
SAME_WITHIN = 0.25

extra_install = rust_environment_cost.min - python_install.min
cheaper = min(python_install.min, rust_environment_cost.min)

print()
if abs(extra_install) / cheaper < SAME_WITHIN:
    print(
        f'The two differ by {abs(extra_install):.2f} s, which is within '
        f'{SAME_WITHIN:.0%} of the smaller'
    )
    print('of them. Setting up either environment costs about the same, and which')
    print('one comes out ahead changes between runs of this cell. Installing costs')
    print('nothing worth counting either way; running is where they differ.')
elif extra_install <= 0:
    print(f'Rust costs {-extra_install:.2f} s less to set up, and also runs faster.')
else:
    print(
        f'Rust costs {extra_install:.2f} s more to set up and saves '
        f'{saving:.1f} ms per run,'
    )
    runs = extra_install / (saving / 1000)
    print(f'so it costs less in total after {runs:.0f} runs.')
print()
print('   Both figures assume a warm package cache. A first install on a new')
print('   machine downloads more and takes longer, on either side.')

Python environment, venv plus pip install   2.64 s   (median 2.65 s)
Rust environment, cargo install            2.22 s   (median 2.28 s)

The two differ by 0.43 s, which is within 25% of the smaller
of them. Setting up either environment costs about the same, and which
one comes out ahead changes between runs of this cell. Installing costs
nothing worth counting either way; running is where they differ.

   Both figures assume a warm package cache. A first install on a new
   machine downloads more and takes longer, on either side.


## What the measurements add up to

The last cell collects the figures above into the statements they support, and
adds one derived number: the file count at which the per-file cost overtakes
the cost of starting the program.

It recommends nothing. Which implementation to use is a short rule that turns
on what is already installed rather than on any measurement here, and it lives
in the README.

In [9]:
import textwrap

# The file count at which the two halves of the total are equal: fixed cost in
# milliseconds against per-file cost in microseconds.
crossover = {
    name: count_intercept[name] * 1000 / count_marginal[name]
    for name in ('Python', 'Rust')
}

large_row = results[-1]
large_ratio = large_row.python.min / large_row.rust.min
startup_ratio = python_startup / rust_startup
bound = BOUND if rust_startup < SPAWN_FLOOR * FLOOR_FACTOR else ''

# Each configuration file that cannot yet take its intended form is marked with
# this phrase, and the comment below the marker says what is blocked and what it
# is blocked on. Those comments are read here rather than summarized, because a
# list of the same caveats kept in this notebook would be a second copy to
# maintain, and the copy nobody edits is the one that becomes wrong.
MARKER = 'DEVIATION, blocked on'


def deviation(path: Path) -> str:
    """Return the comment block introduced by ``MARKER`` in ``path``, as one line."""
    # The only read in this notebook that named no encoding, which meant it
    # decoded with whatever the platform default happened to be.
    lines = path.read_text(encoding='utf-8').splitlines()
    for index, line in enumerate(lines):
        if MARKER in line:
            block = []
            append_to_block = block.append
            for follow in lines[index:]:
                stripped = follow.lstrip()
                if not stripped.startswith('#'):
                    break
                append_to_block(stripped.lstrip('#').strip())
            return ' '.join(part for part in block if part)
    return ''


document = [
    '### What was measured',
    '',
    f'- **Starting the program.** Python takes {bound}{startup_ratio:.0f} times as '
    f'long, a difference of {saving:.0f} ms on every run.',
    f'- **Each additional file.** Python takes between {min(size_ratios):.1f} and '
    f'{max(size_ratios):.1f} times as long, depending on how much text the file '
    f'holds. For files the size of the ones here, {RATIO_TEXT}.',
    f'- **One {large.stat().st_size / 1e6:.0f} MB document**, where starting the '
    f'program is a negligible part of the total: Python takes '
    f'{large_ratio:.1f} times as long.',
    f'- **Setting up an environment.** Python {python_install.min:.2f} s, Rust '
    f'{rust_environment_cost.min:.2f} s'
    + (
        ', which is the same to within the accuracy of the measurement.'
        if abs(extra_install) / cheaper < SAME_WITHIN
        else '.'
    ),
    f'- **Where the balance tips.** Below about {crossover["Python"]:.0f} files in '
    f'one run, most of the Python total is starting the program; above it, most is '
    f'the per-file cost. For Rust that point is about '
    f'{crossover["Rust"]:.0f} files.',
    '',
    '### What was not measured',
    '',
    textwrap.fill(
        'Installing a Rust toolchain, which is what `pre-commit` does before it can '
        'build a `language: rust` hook on a machine without cargo. It costs far more '
        'than either number above, and it is the reason the rule in the README turns '
        'on whether cargo is already installed rather than on any figure here.',
        width=86,
    ),
]

pending = []
append_to_pending = pending.append
for name in ('action.yml', '.pre-commit-hooks.yaml', 'pyproject.toml', 'Cargo.toml'):
    note = deviation(REPO / name)
    if note:
        # The two sentences after the marker describe the deviation, and the
        # marker line itself names what the file waits on -- which is the one
        # thing the lead-in below promises, so it is rendered rather than
        # dropped. Only the marker phrase goes, because the heading says it.
        head, _, rest = note.partition('. ')
        blocker = head.removeprefix(MARKER).strip(' ,.')
        body = '. '.join(part for part in rest.split('. ')[:2] if part).rstrip('.')
        append_to_pending(f'**`{name}`** -- {body}. Blocked on {blocker}.')
if not TAGS:
    append_to_pending(
        '**package registries** -- neither PyPI nor crates.io has this package yet, '
        'so both programs were installed from this checkout rather than from a '
        'published release.'
    )

if pending:
    document += [
        '',
        '### These measurements describe the repository as it is now',
        '',
        textwrap.fill(
            'Each file below records that it cannot yet take its intended form, and '
            'what it is waiting on. Until then, what this notebook measured is what '
            'the repository does today rather than what it is designed to do.',
            width=86,
        ),
        '',
    ]
    document += [f'- {note}' for note in pending]

if not AGREE:
    document += [
        '',
        '**The implementations returned different results above. Ignore all of this.**',
    ]

display(Markdown(chr(10).join(document)))

### What was measured

- **Starting the program.** Python takes ≥15 times as long, a difference of 24 ms on every run.
- **Each additional file.** Python takes between 4.9 and 13.8 times as long, depending on how much text the file holds. For files the size of the ones here, 13.8 times or more.
- **One 4 MB document**, where starting the program is a negligible part of the total: Python takes 15.3 times as long.
- **Setting up an environment.** Python 2.64 s, Rust 2.22 s, which is the same to within the accuracy of the measurement.
- **Where the balance tips.** Below about 305 files in one run, most of the Python total is starting the program; above it, most is the per-file cost. For Rust that point is about 109 files.

### What was not measured

Installing a Rust toolchain, which is what `pre-commit` does before it can build a
`language: rust` hook on a machine without cargo. It costs far more than either number
above, and it is the reason the rule in the README turns on whether cargo is already
installed rather than on any figure here.

### These measurements describe the repository as it is now

Each file below records that it cannot yet take its intended form, and what it is
waiting on. Until then, what this notebook measured is what the repository does today
rather than what it is designed to do.

- **`action.yml`** -- This provisions Python and runs the `-py` implementation, which is not the intended shape. The design calls for downloading a prebuilt binary from the release for this tag -- 480 KB at the largest of the six, no toolchain, faster to install as well as to run -- with an `implementation` input taking `auto`, `rust` or `python`, and `pip install` kept as the fallback for a runner with no published binary. Blocked on the work, not on a release.
- **`.pre-commit-hooks.yaml`** -- This file is meant to go away. At `v0.1.0` each pair moves to a mirror repository serving only its own implementation -- `markdown-prose-hooks-py` and `markdown-prose-hooks-rs` -- so a consumer stops cloning roughly 1.4 MB carrying both plus 373 corpus fixtures and the benchmark notebook to get one of them. Blocked on the mirror repositories.